# alphaPhos collapse benchmark — EGF dataset

Goal: validate the new `alphaphos.preprocess.collapse_sites` against Spectronaut's native PTM site reports across three PSM-level input flavours (old v3 schema, modern MS1+MS2, modern MS2-only) and two Spectronaut native pivot outputs (all sites / Class I only).

**Experimental design:** 6 nanoPhos runs, 1000 ng input, 3 × withEGF + 3 × woEGF (single timepoint).

**Reference axis:** SN Class I PTM site report (`sn_classI`). Every other version is scored on (a) site overlap and (b) per-cell quant agreement (Pearson r in log2 space) against `sn_classI`.

## §0 · Imports + paths

In [2]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

from alphaphos.io import read_spectronaut
from alphaphos.preprocess import (
    collapse_sites,
    apply_condition_aware_classI_mask,
    filter_to_top_n_positions,
    META_COLS,
    to_anndata,
)

BENCH = Path('test_data/benchmark')
OUT = Path('test_data/benchmark_output')
OUT.mkdir(exist_ok=True)

FILES = {
    'old':           BENCH / 'EGF_report_old.tsv',
    'new_ms1_ms2':   BENCH / 'EGF_report_new_ms1_ms2.tsv',
    'new_ms2':       BENCH / 'EGF_report_new_ms2.tsv',
    'sn_all':        BENCH / 'EGF_report_sn_out_all_sites.tsv',
    'sn_classI':     BENCH / 'EGF_report_sn_out_classI_sites.tsv',
}
for k, p in FILES.items():
    print(f'{k:14s}  exists={p.exists()}  size={p.stat().st_size/1e6:.1f} MB')


old             exists=True  size=287.4 MB
new_ms1_ms2     exists=True  size=448.3 MB
new_ms2         exists=True  size=430.7 MB
sn_all          exists=True  size=19.8 MB
sn_classI       exists=True  size=8.8 MB


## §1 · Confirm column structure (no heavy loads)

Three long-format inputs differ in which quant columns they expose. The two SN pivot files share structure but differ only in whether Spectronaut pre-applied the 0.75 Class I filter.

In [3]:
TAB = chr(9)

def peek_cols(path):
    with open(path, 'r', encoding='utf-8') as fh:
        return fh.readline().strip().split(TAB)

for k, p in FILES.items():
    cols = peek_cols(p)
    print()
    print(f'=== {k} ({len(cols)} cols) ===')
    quant_like = [c for c in cols if re.search(r'(Quantity|Intensity|PTM[.]SiteProbability)', c, re.I)]
    for c in quant_like:
        print(f'  {c}')



=== old (15 cols) ===
  PEP.Quantity
  EG.TotalQuantity (Settings)

=== new_ms1_ms2 (33 cols) ===
  PEP.Quantity
  EG.TotalQuantity (Settings)
  FG.MS1Quantity
  FG.MS1RawQuantity
  FG.MS2Quantity
  FG.MS2RawQuantity
  FG.Quantity

=== new_ms2 (31 cols) ===
  PEP.Quantity
  EG.TotalQuantity (Settings)
  FG.MS2Quantity
  FG.MS2RawQuantity
  FG.Quantity

=== sn_all (20 cols) ===
  [1] 20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_01.raw.PTM.SiteProbability
  [2] 20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_02.raw.PTM.SiteProbability
  [3] 20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_03.raw.PTM.SiteProbability
  [4] 20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_woEGF_1000ng_01.raw.PTM.SiteProbability
  [5] 20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_woEGF_1000ng_02.raw.PTM.SiteProbability
  [6] 20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_woEGF_1000ng_03.raw.PTM.SiteProbability
  [1] 2

## §2 · Sample → condition mapping

Samples encoded in `R.FileName` as `..._{withEGF|woEGF}_1000ng_{01-03}`. Two conditions × 3 replicates = 6 samples total.

In [4]:
# Pull just R.FileName from the smallest long-format file (fast)
TAB = chr(9)
_df_for_samples = pd.read_csv(FILES['new_ms2'], sep=TAB, usecols=['R.FileName'])
samples = sorted(_df_for_samples['R.FileName'].unique())

def condition_of(fname):
    if 'withEGF' in fname:
        return 'withEGF'
    if 'woEGF' in fname:
        return 'woEGF'
    return 'UNKNOWN'

condition_df = pd.DataFrame({
    'sample':    samples,
    'condition': [condition_of(s) for s in samples],
})
assert (condition_df['condition'] != 'UNKNOWN').all(), 'unmapped sample'
print(condition_df.to_string(index=False))
print()
print('Counts per condition:')
print(condition_df['condition'].value_counts())


                                                                  sample condition
20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_01   withEGF
20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_02   withEGF
20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_03   withEGF
  20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_woEGF_1000ng_01     woEGF
  20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_woEGF_1000ng_02     woEGF
  20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_woEGF_1000ng_03     woEGF

Counts per condition:
condition
withEGF    3
woEGF      3
Name: count, dtype: int64


## §3 · alphaPhos collapse on 4 PSM variants

For each of the 4 PSM inputs we run **one** `PeptideCollapse` call with
`localization_strategy='global_max'` and `cutoff=0` (permissive) to produce
`sites_raw` + `loc_per_run`, then apply `apply_condition_aware_classI_mask`
to get `sites_classI`. Both versions are cached to parquet under
`test_data/benchmark_output/` so subsequent analysis cells don't re-collapse.

The 4 PSM variants:

| Label | File | `quant_level` |
|---|---|---|
| `aphos_old` | `EGF_report_old.tsv` | `"auto"` (v3 schema; only `EG.TotalQuantity (Settings)`) |
| `aphos_new_MS1` | `EGF_report_new_ms1_ms2.tsv` | `"MS1"` |
| `aphos_new_MS2` | `EGF_report_new_ms1_ms2.tsv` | `"MS2"` |
| `aphos_only_MS2` | `EGF_report_new_ms2.tsv` | `"MS2"` |

**Caching:** if both `{label}_raw.parquet` and `{label}_classI.parquet`
exist, the variant is skipped. To force a re-run, delete those files.


In [5]:
import time

# Per-variant config: (label, file_key, quant_level)
PSM_CONFIGS = [
    ('aphos_old',       'old',         'auto'),
    ('aphos_new_MS1',   'new_ms1_ms2', 'MS1'),
    ('aphos_new_MS2',   'new_ms1_ms2', 'MS2'),
    ('aphos_only_MS2',  'new_ms2',     'MS2'),
]

# Collapse parameters held constant across all variants (apples-to-apples)
COLLAPSE_KWARGS = dict(
    cutoff=0.0,
    collapse_level='PG',
    aggregation_method='median',
    localization_strategy='global_max',
    noise_floor_filter=True,
    add_kinase_sequences=False,  # skip kinase windows for the collapse benchmark
)
CLASSI_KWARGS = dict(
    classI_cutoff=0.75,
    condition_threshold=0.50,
    drop_all_nan=True,
)

def run_one(label, file_key, quant_level):
    raw_path = OUT / f'{label}_raw.parquet'
    classI_path = OUT / f'{label}_classI.parquet'
    loc_path = OUT / f'{label}_loc_per_run.parquet'

    if raw_path.exists() and classI_path.exists() and loc_path.exists():
        print(f'  [{label}] cached, skipping')
        return

    src = FILES[file_key]
    t0 = time.time()
    print(f'  [{label}] reading {src.name} ...')
    df = read_spectronaut(
        src, quant_level=quant_level,
        drop_decoys=True, pg_qvalue_max=0.01,
        top_n_attribution=True,  # validated default
    )
    print(f'    {len(df):,} rows after read+top-N ({time.time()-t0:.1f}s)')

    t0 = time.time()
    print(f'  [{label}] PeptideCollapse (global_max, cutoff=0) ...')
    sites_raw, loc_per_run = collapse_sites(df, **COLLAPSE_KWARGS)
    print(f'    {len(sites_raw):,} sites ({time.time()-t0:.1f}s)')

    sites_raw.to_parquet(raw_path)
    loc_per_run.to_parquet(loc_path)

    t0 = time.time()
    print(f'  [{label}] applying condition_aware Class-I mask ...')
    sites_classI, _ = apply_condition_aware_classI_mask(
        df_sites=sites_raw,
        loc_per_run=loc_per_run,
        sample_to_condition=condition_df,
        return_decision_table=True,
        **CLASSI_KWARGS,
    )
    print(f'    {len(sites_classI):,} sites after mask ({time.time()-t0:.1f}s)')
    sites_classI.to_parquet(classI_path)

    # Free memory before the next variant
    del df, sites_raw, loc_per_run, sites_classI

for label, file_key, quant_level in PSM_CONFIGS:
    print()
    print(f'==== {label} ====')
    run_one(label, file_key, quant_level)

print()
print('Done. Output files:')
for f in sorted(OUT.glob('aphos_*.parquet')):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')



==== aphos_old ====
  [aphos_old] cached, skipping

==== aphos_new_MS1 ====
  [aphos_new_MS1] cached, skipping

==== aphos_new_MS2 ====
  [aphos_new_MS2] cached, skipping

==== aphos_only_MS2 ====
  [aphos_only_MS2] cached, skipping

Done. Output files:
  aphos_new_MS1_classI.parquet  (2.1 MB)
  aphos_new_MS1_loc_per_run.parquet  (0.7 MB)
  aphos_new_MS1_raw.parquet  (2.7 MB)
  aphos_new_MS2_classI.parquet  (2.3 MB)
  aphos_new_MS2_loc_per_run.parquet  (0.7 MB)
  aphos_new_MS2_raw.parquet  (2.8 MB)
  aphos_old_classI.parquet  (2.3 MB)
  aphos_old_loc_per_run.parquet  (0.7 MB)
  aphos_old_raw.parquet  (2.8 MB)
  aphos_only_MS2_classI.parquet  (2.3 MB)
  aphos_only_MS2_loc_per_run.parquet  (0.7 MB)
  aphos_only_MS2_raw.parquet  (2.8 MB)


In [6]:
# Quick site-count summary across the 8 alphaPhos outputs

rows = []
for label, _, _ in PSM_CONFIGS:
    for variant in ('raw', 'classI'):
        path = OUT / f'{label}_{variant}.parquet'
        if not path.exists():
            continue
        d = pd.read_parquet(path)
        sample_cols = [c for c in d.columns if c not in META_COLS]
        block = d[sample_cols]
        rows.append({
            'label': label,
            'variant': variant,
            'n_sites': len(d),
            'cells_total': block.size,
            'cells_present': int(block.notna().sum().sum()),
            'completeness_pct': round(block.notna().sum().sum() / max(1, block.size) * 100, 2),
        })

summary_alphaphos = pd.DataFrame(rows)
print(summary_alphaphos.to_string(index=False))


         label variant  n_sites  cells_total  cells_present  completeness_pct
     aphos_old     raw    42558       255348         154958             60.69
     aphos_old  classI    34248       205488         123312             60.01
 aphos_new_MS1     raw    42558       255348         142588             55.84
 aphos_new_MS1  classI    32408       194448         116102             59.71
 aphos_new_MS2     raw    42558       255348         154958             60.69
 aphos_new_MS2  classI    34248       205488         123312             60.01
aphos_only_MS2     raw    42558       255348         154958             60.69
aphos_only_MS2  classI    34248       205488         123312             60.01


## §4 · Parse Spectronaut pivot outputs

The two SN pivot files (`sn_all`, `sn_classI`) live in a different shape than
alphaPhos output:

- Wide format: per-sample `PTM.SiteProbability` and `PTM.Quantity` columns
- Linear-space quants (we need log2 to match alphaPhos)
- Per-cell `"Filtered"` strings where Spectronaut censored a cell below the
  loc cutoff (only in `sn_classI`)
- Multiplicity not capped at 3 (alphaPhos caps; we mirror that here)
- Single `PTM.ProteinId` per row (alphaPhos uses ProteinGroup with semicolons)

We standardize both to alphaPhos's site-matrix shape:

- Index: `match_key = (PTM.ProteinId, PTM.SiteAA, PTM.SiteLocation, mult_capped)`
- Sample columns: cleaned `R.FileName` values matching the alphaPhos column names
- log2-transformed quants, NaN for `"Filtered"` and zeros

The matching key uses single ProteinId (not ProteinGroup) since Spectronaut splits
shared protein groups into separate rows. For alphaPhos matching we'll parse our
`PTM_Collapse_key` into the same 4-tuple in §5.


In [7]:
import re

def _sample_from_sn_col(col):
    # '[1] 20250729_..._withEGF_1000ng_01.raw.PTM.Quantity' -> '20250729_..._withEGF_1000ng_01'
    m = re.match(r'^\[\d+\]\s+(.+?)\.raw\.PTM\.\w+$', col)
    return m.group(1) if m else col


def _mk(prot, aa, pos, mult):
    # Single-string match key. Components contain no '|' so this round-trips cleanly.
    return f'{prot}|{aa}|{int(pos)}|{int(mult)}'


def parse_sn_pivot(path):
    """Read a Spectronaut PTM site pivot report into a wide DataFrame keyed
    by a stringified match_key, with cleaned sample column names and log2 quants.
    Returns (sites_wide, loc_per_run_wide).
    """
    raw = pd.read_csv(path, sep=chr(9), low_memory=False)
    raw = raw.loc[raw['PTM.ModificationTitle'] == 'Phospho (STY)'].copy()

    prob_cols = [c for c in raw.columns if 'PTM.SiteProbability' in c]
    quant_cols = [c for c in raw.columns if c.endswith('PTM.Quantity')]

    raw['mult_capped'] = raw['PTM.Multiplicity'].clip(upper=3).astype(int)
    raw['match_key'] = [
        _mk(p, a, l, m)
        for p, a, l, m in zip(raw['PTM.ProteinId'], raw['PTM.SiteAA'],
                              raw['PTM.SiteLocation'], raw['mult_capped'])
    ]

    def _coerce(s):
        return pd.to_numeric(s.replace('Filtered', np.nan), errors='coerce')

    sample_quant_map = {c: _sample_from_sn_col(c) for c in quant_cols}
    sample_prob_map  = {c: _sample_from_sn_col(c) for c in prob_cols}

    quant_lin = raw[quant_cols].apply(_coerce).rename(columns=sample_quant_map)
    prob = raw[prob_cols].apply(_coerce).rename(columns=sample_prob_map)
    quant_lin['match_key'] = raw['match_key'].values
    prob['match_key'] = raw['match_key'].values

    # Merge duplicate match_keys (from M4/M5 -> M3 capping)
    quant_lin = quant_lin.groupby('match_key', sort=False).sum(min_count=1)
    prob = prob.groupby('match_key', sort=False).max()

    sites_wide = np.log2(quant_lin.replace(0, np.nan))
    return sites_wide, prob


print('Parsing sn_all ...')
sn_all_sites, sn_all_loc = parse_sn_pivot(FILES['sn_all'])
print(f'  {len(sn_all_sites):,} sites, {sn_all_sites.shape[1]} samples')

print('Parsing sn_classI ...')
sn_classI_sites, sn_classI_loc = parse_sn_pivot(FILES['sn_classI'])
print(f'  {len(sn_classI_sites):,} sites, {sn_classI_sites.shape[1]} samples')

# (No parquet cache: index is now a plain string, but SN parsing is fast
#  enough that re-running §4 after a kernel restart is trivial anyway.)
print()
print('Sample column names (first 3 of each):')
print('  sn_all     :', list(sn_all_sites.columns)[:3])
print('  sn_classI  :', list(sn_classI_sites.columns)[:3])
print('  example key:', sn_all_sites.index[0])


Parsing sn_all ...
  66,410 sites, 6 samples
Parsing sn_classI ...
  34,636 sites, 6 samples

Sample column names (first 3 of each):
  sn_all     : ['20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_01', '20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_02', '20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_03']
  sn_classI  : ['20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_01', '20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_02', '20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_new_withEGF_1000ng_03']
  example key: A0A087WUV0|T|112|1


In [8]:
KEY_RE = re.compile(
    r'^(?P<prot>[^~]+)~(?P<gene>[^_]*)_(?P<aa>[A-Z])(?P<pos>\d+)_M(?P<mult>\d+)$'
)


def alpha_to_matchkey_frame(sites_df):
    """Convert an alphaphos site matrix into a wide DataFrame indexed by
    a stringified match key 'ProteinId|AA|position|mult_capped' for direct
    comparison against parsed SN pivot output.
    """
    parsed = sites_df['PTM_Collapse_key'].astype(str).str.extract(KEY_RE)
    first_prot = parsed['prot'].str.split(';').str[0]
    mult_capped = parsed['mult'].astype(int).clip(upper=3)

    keys = [
        f'{p}|{a}|{int(l)}|{int(m)}'
        for p, a, l, m in zip(first_prot, parsed['aa'], parsed['pos'].astype(int), mult_capped)
    ]

    sample_cols = [c for c in sites_df.columns if c not in META_COLS]
    block = sites_df[sample_cols].copy()
    block.index = pd.Index(keys, name='match_key')
    # If multiple PTM_Collapse_keys map to the same match_key, keep the first
    block = block[~block.index.duplicated(keep='first')]
    return block


APHOS_LABELS = ['aphos_old', 'aphos_new_MS1', 'aphos_new_MS2', 'aphos_only_MS2']
APHOS_VARIANTS = ['raw', 'classI']

aphos_matched = {}
for label in APHOS_LABELS:
    for variant in APHOS_VARIANTS:
        key = f'{label}_{variant}'
        df = pd.read_parquet(OUT / f'{label}_{variant}.parquet')
        aphos_matched[key] = alpha_to_matchkey_frame(df)
        print(f'{key:>26s}: {len(aphos_matched[key]):>6,} sites, {aphos_matched[key].shape[1]} samples')

print()
print(f'sn_all      sites: {len(sn_all_sites):,}, samples: {sn_all_sites.shape[1]}')
print(f'sn_classI   sites: {len(sn_classI_sites):,}, samples: {sn_classI_sites.shape[1]}')
print()
print('Example alphaphos key :', list(aphos_matched.values())[0].index[0])
print('Example sn_classI key :', sn_classI_sites.index[0])


             aphos_old_raw: 42,558 sites, 6 samples
          aphos_old_classI: 34,248 sites, 6 samples
         aphos_new_MS1_raw: 42,558 sites, 6 samples
      aphos_new_MS1_classI: 32,408 sites, 6 samples
         aphos_new_MS2_raw: 42,558 sites, 6 samples
      aphos_new_MS2_classI: 34,248 sites, 6 samples
        aphos_only_MS2_raw: 42,558 sites, 6 samples
     aphos_only_MS2_classI: 34,248 sites, 6 samples

sn_all      sites: 66,410, samples: 6
sn_classI   sites: 34,636, samples: 6

Example alphaphos key : A0A087WUV0|S|118|1
Example sn_classI key : A0A087WUV0|S|118|1


## §5 · Cross-version comparison

10 versions total in `aphos_matched` + `sn_all_sites` + `sn_classI_sites`,
all keyed by stringified `'ProteinId|AA|position|mult_capped'`.

Reference axis: **`sn_classI`** (Spectronaut's native PTM site report with the
default Class-I filter applied — the published-tool gold standard).

We compute three views:

1. **Pairwise site overlap (Jaccard)** — 10×10 heatmap of which versions
   identify the same sites
2. **Per-cell quant agreement** vs `sn_classI` — Pearson r in log2 space on
   the cell-by-cell intersection of keys × samples, plus mean log2 diff
3. **Ranked summary table** — each version scored against `sn_classI`

The structural caveat noted above (alphaPhos one row per ProteinGroup vs SN
one row per ProteinId for shared protein groups) means alphaPhos vs SN site
overlap will be capped below 1.0 by the fraction of shared-group sites.
That's correct, not a bug.


In [9]:
# Unified dict of all 10 versions, all keyed by the same string format
VERSIONS = {**aphos_matched,
            'sn_all':     sn_all_sites,
            'sn_classI':  sn_classI_sites}

# Standardize sample column ordering across versions
sample_order = sorted(condition_df['sample'].tolist())
for k in list(VERSIONS):
    VERSIONS[k] = VERSIONS[k].reindex(columns=sample_order)

# Confirm shape consistency
print('Version  ->  (n_sites, n_samples)')
for k, v in VERSIONS.items():
    print(f'  {k:>26s}: {v.shape}')


Version  ->  (n_sites, n_samples)
               aphos_old_raw: (42558, 6)
            aphos_old_classI: (34248, 6)
           aphos_new_MS1_raw: (42558, 6)
        aphos_new_MS1_classI: (32408, 6)
           aphos_new_MS2_raw: (42558, 6)
        aphos_new_MS2_classI: (34248, 6)
          aphos_only_MS2_raw: (42558, 6)
       aphos_only_MS2_classI: (34248, 6)
                      sn_all: (66410, 6)
                   sn_classI: (34636, 6)


In [10]:
# 1) Pairwise site overlap (Jaccard) across all 10 versions

key_sets = {k: set(v.index) for k, v in VERSIONS.items()}

names = list(VERSIONS.keys())
J = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        sa, sb = key_sets[a], key_sets[b]
        J.loc[a, b] = round(len(sa & sb) / max(1, len(sa | sb)), 3)

print('Jaccard site-overlap (10x10):')
print(J.to_string())


Jaccard site-overlap (10x10):
                       aphos_old_raw  aphos_old_classI  aphos_new_MS1_raw  aphos_new_MS1_classI  aphos_new_MS2_raw  aphos_new_MS2_classI  aphos_only_MS2_raw  aphos_only_MS2_classI  sn_all  sn_classI
aphos_old_raw                  1.000             0.805              1.000                 0.762              1.000                 0.805               1.000                  0.805   0.641      0.796
aphos_old_classI               0.805             1.000              0.805                 0.946              0.805                 1.000               0.805                  1.000   0.516      0.980
aphos_new_MS1_raw              1.000             0.805              1.000                 0.762              1.000                 0.805               1.000                  0.805   0.641      0.796
aphos_new_MS1_classI           0.762             0.946              0.762                 1.000              0.762                 0.946               0.762                  

In [11]:
# 2) Per-cell quant agreement vs sn_classI
#    Pearson r on log2 values for the intersection of keys x samples.

REF = 'sn_classI'
ref = VERSIONS[REF]

def agree_vs_ref(version_label):
    v = VERSIONS[version_label]
    shared_keys = ref.index.intersection(v.index)
    shared_cols = ref.columns.intersection(v.columns)
    if len(shared_keys) == 0 or len(shared_cols) == 0:
        return None
    a = v.loc[shared_keys, shared_cols]
    b = ref.loc[shared_keys, shared_cols]

    # Long form: one row per (site, sample) cell with both values non-NaN
    both = pd.DataFrame({
        'a': a.values.flatten(),
        'b': b.values.flatten(),
    }).dropna()
    if len(both) < 50:
        return None

    pearson = float(both['a'].corr(both['b']))
    spearman = float(both['a'].corr(both['b'], method='spearman'))
    diff = both['a'] - both['b']
    return {
        'n_shared_keys': len(shared_keys),
        'n_paired_cells': len(both),
        'pearson_log2': round(pearson, 4),
        'spearman_log2': round(spearman, 4),
        'mean_log2_diff': round(float(diff.mean()), 4),
        'median_log2_diff': round(float(diff.median()), 4),
        'sd_log2_diff': round(float(diff.std()), 4),
        'p05_log2_diff': round(float(diff.quantile(0.05)), 4),
        'p95_log2_diff': round(float(diff.quantile(0.95)), 4),
        'pct_within_0.1_log2': round(float((diff.abs() < 0.1).mean() * 100), 2),
        'pct_within_0.5_log2': round(float((diff.abs() < 0.5).mean() * 100), 2),
    }

rows = []
for k in names:
    if k == REF:
        continue
    res = agree_vs_ref(k)
    rows.append({'version': k, **(res if res else {})})

agreement_vs_classI = pd.DataFrame(rows)
print(f'Cell-level quant agreement vs {REF!r}:')
print(agreement_vs_classI.to_string(index=False))


Cell-level quant agreement vs 'sn_classI':
              version  n_shared_keys  n_paired_cells  pearson_log2  spearman_log2  mean_log2_diff  median_log2_diff  sd_log2_diff  p05_log2_diff  p95_log2_diff  pct_within_0.1_log2  pct_within_0.5_log2
        aphos_old_raw          34210          119162        0.9192         0.9298         -0.5249           -0.0121        1.0668        -2.8279         0.0414                63.43                65.99
     aphos_old_classI          34091          118777        0.9192         0.9298         -0.5249           -0.0121        1.0670        -2.8281         0.0414                63.43                65.99
    aphos_new_MS1_raw          34210          112278        0.7939         0.8052          4.6811            4.6920        1.5877         2.1205         7.1761                 0.12                 0.66
 aphos_new_MS1_classI          32255          111925        0.7938         0.8052          4.6812            4.6923        1.5882         2.1191     

In [12]:
# 3) Final ranking table — combine overlap + quant agreement into one view

ranking = agreement_vs_classI.copy()
ranking['n_sites'] = ranking['version'].map(lambda k: len(VERSIONS[k]))
ranking['jaccard_vs_classI'] = ranking['version'].map(lambda k: J.loc[k, REF])

# Order columns and rank by pearson r descending (the most defensible single metric)
order = ['version', 'n_sites', 'jaccard_vs_classI', 'n_shared_keys',
         'n_paired_cells', 'pearson_log2', 'spearman_log2',
         'mean_log2_diff', 'median_log2_diff', 'sd_log2_diff',
         'pct_within_0.1_log2', 'pct_within_0.5_log2']
ranking = ranking[order].sort_values('pearson_log2', ascending=False).reset_index(drop=True)
ranking.insert(0, 'rank', range(1, len(ranking) + 1))

print('Cross-version ranking by Pearson r vs sn_classI (higher is better):')
print(ranking.to_string(index=False))


Cross-version ranking by Pearson r vs sn_classI (higher is better):
 rank               version  n_sites  jaccard_vs_classI  n_shared_keys  n_paired_cells  pearson_log2  spearman_log2  mean_log2_diff  median_log2_diff  sd_log2_diff  pct_within_0.1_log2  pct_within_0.5_log2
    1                sn_all    66410              0.520          34588          120665        0.9196         0.9123          0.4246            0.0138        1.0612                68.74                80.14
    2         aphos_old_raw    42558              0.796          34210          119162        0.9192         0.9298         -0.5249           -0.0121        1.0668                63.43                65.99
    3      aphos_old_classI    34248              0.980          34091          118777        0.9192         0.9298         -0.5249           -0.0121        1.0670                63.43                65.99
    4  aphos_new_MS2_classI    34248              0.980          34091          118777        0.9192        

## §6 · Deep diff — `aphos_new_MS2_classI` vs `sn_classI`

We have established that alphaPhos + MS2 + condition-aware classI matches SN
Class I at Jaccard 0.98 / Pearson r 0.92 / mean log2 diff -0.52. Now we
characterize the residual differences.

1. **Site-set differences** — which sites are in only one of the two?
2. **Cell-level NaN patterns** — for shared sites, are there cells where
   only one side has a value?
3. **Top disagreement sites** — the worst per-site log2 diffs
4. **Stratifications** — how does the diff depend on multiplicity, S/T/Y,
   localization probability, quant magnitude?
5. **Bland-Altman** — visualize the systematic vs random components of disagreement.


In [13]:
# Focus pair
APHOS_KEY = 'aphos_new_MS2_classI'  # old/new_MS2/only_MS2 are identical
SN_KEY    = 'sn_classI'

A = VERSIONS[APHOS_KEY]
B = VERSIONS[SN_KEY]

a_only = sorted(set(A.index) - set(B.index))
b_only = sorted(set(B.index) - set(A.index))
shared = sorted(set(A.index) & set(B.index))

print(f'aphos_new_MS2_classI sites: {len(A):,}')
print(f'sn_classI            sites: {len(B):,}')
print()
print(f'  shared (in both):     {len(shared):,}')
print(f'  aphos-only:           {len(a_only):,}')
print(f'  sn-only:              {len(b_only):,}')
print()
print('Examples of aphos-only keys (first 10):')
for k in a_only[:10]:
    print(f'  {k}')
print()
print('Examples of sn-only keys (first 10):')
for k in b_only[:10]:
    print(f'  {k}')


aphos_new_MS2_classI sites: 34,248
sn_classI            sites: 34,636

  shared (in both):     34,091
  aphos-only:           157
  sn-only:              545

Examples of aphos-only keys (first 10):
  A0FGR8|T|701|2
  A8CG34|S|325|3
  A8CG34|S|328|3
  O00470|S|257|2
  O00712|S|311|1
  O14730|Y|122|3
  O15085|S|1458|2
  O43353|S|361|1
  O75122|S|459|3
  O75369|S|2135|1

Examples of sn-only keys (first 10):
  A0A0B4J2F2|S|435|1
  A0A0B4J2F2|S|437|1
  A0A0B4J2F2|S|534|1
  A0A0B4J2F2|S|575|1
  A0A2R8YFR7|S|1125|1
  A0A2R8YFR7|S|1126|1
  A0A2R8YFR7|S|11|1
  A0A2R8YFR7|S|15|1
  A0A2R8YFR7|S|445|1
  A0A2R8YFR7|S|450|1


In [14]:
# Site-set differences — break down by multiplicity and amino acid
def key_components(keys):
    rows = []
    for k in keys:
        parts = k.split('|')
        if len(parts) == 4:
            rows.append({'protein': parts[0], 'aa': parts[1],
                         'pos': int(parts[2]), 'mult': int(parts[3])})
    return pd.DataFrame(rows)

shared_df  = key_components(shared).assign(set='shared')
a_only_df  = key_components(a_only).assign(set='aphos_only')
b_only_df  = key_components(b_only).assign(set='sn_only')
all_keys_df = pd.concat([shared_df, a_only_df, b_only_df], ignore_index=True)

print('--- by multiplicity ---')
print(pd.crosstab(all_keys_df['mult'], all_keys_df['set'], margins=True))
print()
print('--- by amino acid ---')
print(pd.crosstab(all_keys_df['aa'], all_keys_df['set'], margins=True))
print()
print('--- per-set distribution by multiplicity (column %) ---')
print((pd.crosstab(all_keys_df['mult'], all_keys_df['set'], normalize='columns') * 100).round(1))


--- by multiplicity ---
set   aphos_only  shared  sn_only    All
mult                                    
1             67   29508      491  30066
2             56    3319       24   3399
3             34    1264       30   1328
All          157   34091      545  34793

--- by amino acid ---
set  aphos_only  shared  sn_only    All
aa                                     
S           108   25577      355  26040
T            37    6553      135   6725
Y            12    1961       55   2028
All         157   34091      545  34793

--- per-set distribution by multiplicity (column %) ---
set   aphos_only  shared  sn_only
mult                             
1           42.7    86.6     90.1
2           35.7     9.7      4.4
3           21.7     3.7      5.5


In [15]:
# Cell-level NaN patterns on the shared site set
sample_cols = condition_df['sample'].tolist()
A_shared = A.loc[shared, sample_cols]
B_shared = B.loc[shared, sample_cols]

a_present = A_shared.notna()
b_present = B_shared.notna()

n_both    = int((a_present & b_present).sum().sum())
n_a_only  = int((a_present & ~b_present).sum().sum())
n_b_only  = int((~a_present & b_present).sum().sum())
n_neither = int((~a_present & ~b_present).sum().sum())
n_total = a_present.size

print(f'Shared sites: {len(shared):,}, samples: {a_present.shape[1]}, total cells: {n_total:,}')
print()
print(f'  cell present in BOTH      : {n_both:>7,}  ({n_both/n_total*100:5.1f}%)')
print(f'  cell present ONLY in aphos: {n_a_only:>7,}  ({n_a_only/n_total*100:5.1f}%)')
print(f'  cell present ONLY in SN   : {n_b_only:>7,}  ({n_b_only/n_total*100:5.1f}%)')
print(f'  cell NaN in BOTH          : {n_neither:>7,}  ({n_neither/n_total*100:5.1f}%)')
print()
print('Interpretation:')
print('  aphos-only cells: aphos kept a measurement SN censored as Filtered')
print('  (per-condition majority-keep vs per-cell threshold; expected).')
print('  SN-only cells: SN has a value where aphos has NaN (less expected; investigate).')


Shared sites: 34,091, samples: 6, total cells: 204,546

  cell present in BOTH      : 118,777  ( 58.1%)
  cell present ONLY in aphos:   4,186  (  2.0%)
  cell present ONLY in SN   :      86  (  0.0%)
  cell NaN in BOTH          :  81,497  ( 39.8%)

Interpretation:
  aphos-only cells: aphos kept a measurement SN censored as Filtered
  (per-condition majority-keep vs per-cell threshold; expected).
  SN-only cells: SN has a value where aphos has NaN (less expected; investigate).


In [16]:
# Per-site median log2 diff
diff_long = (A_shared - B_shared).stack(dropna=False)
diff_long = diff_long.dropna().rename('log2_diff').reset_index()
diff_long.columns = ['match_key', 'sample', 'log2_diff']

per_site = diff_long.groupby('match_key').agg(
    n_cells=('log2_diff', 'size'),
    median_log2_diff=('log2_diff', 'median'),
    max_abs_log2_diff=('log2_diff', lambda s: s.abs().max()),
)

components = key_components(per_site.index.tolist()).assign(match_key=per_site.index.tolist()).set_index('match_key')
per_site = per_site.join(components)

print('Top 20 sites by |median log2 diff| (alphaphos - SN classI):')
top = per_site.reindex(per_site['median_log2_diff'].abs().sort_values(ascending=False).index).head(20)
print(top.to_string())
print()
print('Per-site median diff distribution:')
print(per_site['median_log2_diff'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3))


C:\Users\oliinyk\AppData\Local\Temp\ipykernel_44132\3217008346.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  diff_long = (A_shared - B_shared).stack(dropna=False)


Top 20 sites by |median log2 diff| (alphaphos - SN classI):
                 n_cells  median_log2_diff  max_abs_log2_diff protein aa   pos  mult
match_key                                                                           
P05387|S|105|2         6         -8.886711           9.270543  P05387  S   105     2
P05387|S|102|2         6         -8.886711           9.270543  P05387  S   102     2
P04792|S|82|1          6         -8.728592           8.972718  P04792  S    82     1
P07900|S|231|1         6         -8.262272           9.265578  P07900  S   231     1
Q14934|S|289|1         6         -8.175276           8.490147  Q14934  S   289     1
P29692|S|162|1         6         -7.905789           8.460221  P29692  S   162     1
P20042|S|2|1           6         -7.799852           8.095401  P20042  S     2     1
P11717|S|2409|1        6         -7.650084           7.761210  P11717  S  2409     1
Q8WYP5|S|1232|1        6         -7.540894           8.269068  Q8WYP5  S  1232     1
Q7Z5L

In [17]:
# Stratify cell-level diff by multiplicity, amino acid, and quant magnitude
diff_full = diff_long.copy()
parts_split = diff_full['match_key'].str.split('|', expand=True)
diff_full['aa'] = parts_split[1]
diff_full['mult'] = parts_split[3].astype(int)

sn_long = B_shared.stack(dropna=False).dropna().rename('sn_log2').reset_index()
sn_long.columns = ['match_key', 'sample', 'sn_log2']
diff_full = diff_full.merge(sn_long, on=['match_key', 'sample'], how='left')
diff_full['sn_quant_bin'] = pd.cut(
    diff_full['sn_log2'],
    bins=[-np.inf, 4, 6, 8, 10, 12, np.inf],
    labels=['<4', '4-6', '6-8', '8-10', '10-12', '>12'],
)

def _strat(by):
    g = diff_full.groupby(by, observed=True)['log2_diff']
    return pd.DataFrame({
        'n':              g.size(),
        'mean':           g.mean().round(3),
        'median':         g.median().round(3),
        'p05':            g.quantile(0.05).round(3),
        'p95':            g.quantile(0.95).round(3),
        'pct_within_0.1': (g.apply(lambda s: (s.abs() < 0.1).mean() * 100)).round(2),
    })

print('--- by multiplicity ---')
print(_strat('mult'))
print()
print('--- by amino acid ---')
print(_strat('aa'))
print()
print('--- by SN classI quant magnitude (log2) ---')
print(_strat('sn_quant_bin'))


--- by multiplicity ---
          n   mean  median    p05    p95  pct_within_0.1
mult                                                    
1     97873 -0.510  -0.012 -2.796  0.042           64.17
2     15087 -0.609  -0.014 -3.146  0.036           61.26
3      5817 -0.561  -0.017 -2.536  0.040           56.51

--- by amino acid ---
        n   mean  median    p05    p95  pct_within_0.1
aa                                                    
S   94191 -0.569  -0.014 -2.972  0.040           61.21
T   19230 -0.415  -0.009 -2.443  0.043           68.97
Y    5356 -0.142  -0.003 -1.011  0.046           82.52

--- by SN classI quant magnitude (log2) ---
                  n   mean  median    p05    p95  pct_within_0.1
sn_quant_bin                                                    
<4             2047  0.101   0.001 -0.094  0.658           88.28
4-6           12167  0.041   0.001 -0.087  0.131           90.13
6-8           30964 -0.073  -0.002 -0.997  0.054           84.61
8-10          33877 -0.

C:\Users\oliinyk\AppData\Local\Temp\ipykernel_44132\654248586.py:7: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  sn_long = B_shared.stack(dropna=False).dropna().rename('sn_log2').reset_index()


In [18]:
# Bland-Altman: mean vs difference (log2 space)
import plotly.express as px

ba = diff_full.dropna(subset=['sn_log2']).copy()
ba['mean_log2'] = ba['sn_log2'] + ba['log2_diff'] / 2

n_plot = min(10000, len(ba))
sub = ba.sample(n=n_plot, random_state=0)

fig = px.scatter(
    sub,
    x='mean_log2', y='log2_diff',
    color=sub['mult'].astype(str),
    opacity=0.35,
    hover_data={'match_key': True, 'mean_log2': ':.2f',
                'log2_diff': ':.3f', 'mult': True, 'aa': True},
    title=f'Bland-Altman (subsample of {n_plot:,} cells) - alphaPhos vs SN Class I',
    labels={'mean_log2': 'mean log2 intensity',
            'log2_diff': 'log2 diff (alphaphos - SN)',
            'color': 'multiplicity'},
)
mean_diff = ba['log2_diff'].mean()
sd_diff = ba['log2_diff'].std()
fig.add_hline(y=mean_diff, line_dash='dash', annotation_text=f'mean = {mean_diff:.3f}')
fig.add_hline(y=mean_diff + 1.96*sd_diff, line_dash='dot',
              annotation_text=f'+1.96 SD = {mean_diff + 1.96*sd_diff:.2f}')
fig.add_hline(y=mean_diff - 1.96*sd_diff, line_dash='dot',
              annotation_text=f'-1.96 SD = {mean_diff - 1.96*sd_diff:.2f}')
fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(height=500, width=900, template='plotly_white')
fig.show()


## §6.5 · Test `aggregation_method='consolidate'`

§6 showed the systematic alphaPhos-vs-SN log2 offset is **intensity-dependent**:
near-zero at low quant, grows to ~-1.0 log2 at high quant. The likely cause is
**median vs sum aggregation** of multi-precursor sites:

- alphaPhos default we used: `aggregation_method='median'` (Dublin convention)
- SN's PTM consolidation: `"Linear modeling based"` = the canonical Hogrebe
  ratio-imputation + **sum**. We already ported this as `'consolidate'`.

We re-collapse the same MS2 input with `aggregation_method='consolidate'` and
re-score the result against `sn_classI`. If our diagnosis is right, Pearson r
should rise from 0.92 and the mean log2 diff should drop from -0.52 toward zero.

The simpler `aggregation_method='sum'` is a sanity check (SN's other PTM
consolidation option — plain sum without imputation).


In [19]:
import time

# Re-collapse aphos_only_MS2 (cheapest single-quant file) with both alternative
# aggregations. Skip if cached.

ALT_AGGS = ['consolidate', 'sum']

def _recollapse(agg_method):
    raw_path = OUT / f'aphos_only_MS2_{agg_method}_raw.parquet'
    classI_path = OUT / f'aphos_only_MS2_{agg_method}_classI.parquet'
    if raw_path.exists() and classI_path.exists():
        print(f'  [{agg_method}] cached, skipping')
        return

    t0 = time.time()
    df = read_spectronaut(FILES['new_ms2'], quant_level='MS2',
                          drop_decoys=True, pg_qvalue_max=0.01,
                          top_n_attribution=True)
    print(f'  [{agg_method}] loaded {len(df):,} rows ({time.time()-t0:.1f}s)')

    t0 = time.time()
    sites_raw, loc_per_run = collapse_sites(
        df,
        cutoff=0.0,
        collapse_level='PG',
        aggregation_method=agg_method,
        localization_strategy='global_max',
        noise_floor_filter=True,
        add_kinase_sequences=False,
    )
    print(f'  [{agg_method}] collapsed -> {len(sites_raw):,} sites ({time.time()-t0:.1f}s)')

    sites_classI, _ = apply_condition_aware_classI_mask(
        df_sites=sites_raw, loc_per_run=loc_per_run,
        sample_to_condition=condition_df,
        classI_cutoff=0.75, condition_threshold=0.50,
        drop_all_nan=True, return_decision_table=True,
    )
    sites_raw.to_parquet(raw_path)
    sites_classI.to_parquet(classI_path)
    print(f'  [{agg_method}] classI -> {len(sites_classI):,} sites')

for agg in ALT_AGGS:
    print(f'==== aggregation_method={agg} ====')
    _recollapse(agg)
    print()

print('Done. Output files:')
for agg in ALT_AGGS:
    for v in ('raw', 'classI'):
        f = OUT / f'aphos_only_MS2_{agg}_{v}.parquet'
        if f.exists():
            print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')


==== aggregation_method=consolidate ====
  [consolidate] loaded 240,099 rows (8.3s)


D:\Projects\alphaPhos\src\alphaphos\preprocess\collapse.py:1474: RuntimeWarning:

All-NaN slice encountered



  [consolidate] collapsed -> 42,558 sites (16.6s)
  [consolidate] classI -> 26,450 sites

==== aggregation_method=sum ====
  [sum] loaded 240,099 rows (8.3s)
  [sum] collapsed -> 42,558 sites (9.1s)
  [sum] classI -> 34,248 sites

Done. Output files:
  aphos_only_MS2_consolidate_raw.parquet  (2.6 MB)
  aphos_only_MS2_consolidate_classI.parquet  (1.9 MB)
  aphos_only_MS2_sum_raw.parquet  (2.8 MB)
  aphos_only_MS2_sum_classI.parquet  (2.3 MB)


In [20]:
# Compare consolidate / sum / median (the original) against sn_classI
ref = VERSIONS['sn_classI']
sample_cols_ref = condition_df['sample'].tolist()


def score_vs_classI(label, sites_df):
    keyed = alpha_to_matchkey_frame(sites_df)
    keyed = keyed.reindex(columns=sample_cols_ref)
    shared_keys = ref.index.intersection(keyed.index)
    a = keyed.loc[shared_keys]
    b = ref.loc[shared_keys]
    both = pd.DataFrame({'a': a.values.flatten(), 'b': b.values.flatten()}).dropna()
    if len(both) < 50:
        return None
    diff = both['a'] - both['b']
    return {
        'config':       label,
        'n_sites':      len(keyed),
        'shared_keys':  len(shared_keys),
        'paired_cells': len(both),
        'pearson_log2': round(float(both['a'].corr(both['b'])), 4),
        'mean_log2_diff':   round(float(diff.mean()), 4),
        'median_log2_diff': round(float(diff.median()), 4),
        'sd_log2_diff':     round(float(diff.std()), 4),
        'p05_diff':         round(float(diff.quantile(0.05)), 4),
        'p95_diff':         round(float(diff.quantile(0.95)), 4),
        'pct_within_0.1':   round(float((diff.abs() < 0.1).mean() * 100), 2),
        'pct_within_0.5':   round(float((diff.abs() < 0.5).mean() * 100), 2),
    }


rows = []
# Baseline: the median version we already have
rows.append(score_vs_classI('median   (already in §5)', pd.read_parquet(OUT / 'aphos_only_MS2_classI.parquet')))
for agg in ALT_AGGS:
    f = OUT / f'aphos_only_MS2_{agg}_classI.parquet'
    rows.append(score_vs_classI(f'{agg:>9s}', pd.read_parquet(f)))

agg_compare = pd.DataFrame(rows)
print('Cell-level agreement vs sn_classI across alphaPhos aggregation methods:')
print(agg_compare.to_string(index=False))


Cell-level agreement vs sn_classI across alphaPhos aggregation methods:
                  config  n_sites  shared_keys  paired_cells  pearson_log2  mean_log2_diff  median_log2_diff  sd_log2_diff  p05_diff  p95_diff  pct_within_0.1  pct_within_0.5
median   (already in §5)    34248        34091        118777        0.9192         -0.5249           -0.0121        1.0670   -2.8281    0.0414           63.43           65.99
             consolidate    26450        26378        104880        0.9810          0.1086            0.0046        0.5192   -0.0306    0.7167           84.74           92.79
                     sum    34248        34091        118777        0.9901          0.0625            0.0021        0.3738   -0.0300    0.2669           91.59           96.31


In [21]:
# Re-stratify the consolidate version by SN quant magnitude — does the
# intensity-dependent bias collapse?

best_label = 'aphos_only_MS2_consolidate_classI'
best_df = alpha_to_matchkey_frame(pd.read_parquet(OUT / 'aphos_only_MS2_consolidate_classI.parquet'))
best_df = best_df.reindex(columns=sample_cols_ref)

shared_keys = ref.index.intersection(best_df.index)
A2 = best_df.loc[shared_keys]
B2 = ref.loc[shared_keys]

d_long = (A2 - B2).stack(dropna=False).dropna().rename('log2_diff').reset_index()
d_long.columns = ['match_key', 'sample', 'log2_diff']
b_long = B2.stack(dropna=False).dropna().rename('sn_log2').reset_index()
b_long.columns = ['match_key', 'sample', 'sn_log2']
d_long = d_long.merge(b_long, on=['match_key', 'sample'], how='left')
d_long['sn_quant_bin'] = pd.cut(
    d_long['sn_log2'],
    bins=[-np.inf, 4, 6, 8, 10, 12, np.inf],
    labels=['<4', '4-6', '6-8', '8-10', '10-12', '>12'],
)

g = d_long.groupby('sn_quant_bin', observed=True)['log2_diff']
strat = pd.DataFrame({
    'n':              g.size(),
    'mean':           g.mean().round(3),
    'median':         g.median().round(3),
    'p05':            g.quantile(0.05).round(3),
    'p95':            g.quantile(0.95).round(3),
    'pct_within_0.1': (g.apply(lambda s: (s.abs() < 0.1).mean() * 100)).round(2),
})
print(f'Stratification by SN quant magnitude — {best_label!r}:')
print(strat)


Stratification by SN quant magnitude — 'aphos_only_MS2_consolidate_classI':
                  n   mean  median    p05    p95  pct_within_0.1
sn_quant_bin                                                    
<4             1550  0.201   0.002 -0.061  1.859           87.03
4-6            9908  0.177   0.002 -0.048  1.494           88.26
6-8           26299  0.147   0.003 -0.032  1.136           86.01
8-10          29883  0.110   0.004 -0.030  0.789           82.94
10-12         21444  0.078   0.007 -0.027  0.491           81.99
>12           15796  0.031   0.006 -0.026  0.273           87.34


C:\Users\oliinyk\AppData\Local\Temp\ipykernel_44132\1016108431.py:12: FutureWarning:

The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.

C:\Users\oliinyk\AppData\Local\Temp\ipykernel_44132\1016108431.py:14: FutureWarning:

The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.



## §7 · Final ranking

Single table: each of the 7 versions scored against `sn_classI` on (Jaccard site overlap, Pearson r on shared cells).

*Next cell to be added.*

## §7 · Full downstream pipeline — `aphos_sum_classI` vs `sn_classI`

We have the two best-matching versions:

- **`aphos_only_MS2_sum_classI`** — alphaPhos MS2 + sum aggregation + condition-aware classI
- **`sn_classI`** — Spectronaut native PTM Class-I site report

Question: do they capture the same biology when we run them through the lab's
standard downstream pipeline?

The pipeline is:

1. **Valid-values filter** — keep sites with ≥70% non-NaN replicates in at
   least one condition (lab convention)
2. **KNN imputation** — fill the remaining sparse NaNs
3. **limma** — withEGF vs woEGF contrast (3 reps each), Rscript subprocess
4. **Compare** — significant-hit overlap, logFC Pearson r on shared features,
   top hits per version

If the two versions capture the same biology we expect Jaccard of significant
hits > 0.85 and Pearson r on logFC > 0.95.


In [22]:
import sys
sys.path.insert(0, r'D:/Projects/Dublin/testscripts/src')
import importlib
# Dublin core.py provides filter_phosphosites + impute_phosphosites
import core as dublin_core
importlib.reload(dublin_core)
from core import filter_phosphosites, impute_phosphosites

# Two versions to compare. Both are already site-level wide matrices.
APHOS_KEY = 'aphos_only_MS2_sum_classI'
SN_KEY    = 'sn_classI'

# Load the alphaphos version (it was generated in §6.5 with aggregation=sum)
aphos_sites = pd.read_parquet(OUT / f'aphos_only_MS2_sum_classI.parquet')
# sn_classI_sites is already in memory from §4

# Bring alphaphos to the same key format we used for comparison in §5
# But the downstream pipeline expects a wide DF with PTM_Collapse_key column,
# so we keep the original alphaphos shape for the filter/impute path.

print(f'aphos sites: {aphos_sites.shape} (rows=sites, cols=samples+meta)')
print(f'sn_classI:   {sn_classI_sites.shape}')

# For the pipeline we need: samples-as-rows, sites-as-cols + a 'condition' col.
# Build the standardized wide-by-sample form for each version.
sample_cols = condition_df['sample'].tolist()
s2c_map = dict(zip(condition_df['sample'], condition_df['condition']))


def to_samples_as_rows(sites_df, key_col=None):
    """Wide DataFrame: rows=samples, cols=sites + 'condition'."""
    if key_col is not None:
        # alphaphos: site identifier is in column 'PTM_Collapse_key'
        block = sites_df.set_index(key_col)[sample_cols].T
    else:
        # SN: site identifier is already the index (string)
        block = sites_df.reindex(columns=sample_cols).T
    block.index.name = 'sample'
    block['condition'] = block.index.map(s2c_map)
    return block


aphos_wide = to_samples_as_rows(aphos_sites, key_col='PTM_Collapse_key')
sn_wide    = to_samples_as_rows(sn_classI_sites, key_col=None)

print(f'\naphos_wide: {aphos_wide.shape}   (rows=samples, cols=sites + condition)')
print(f'sn_wide:    {sn_wide.shape}')
print()
print(aphos_wide[['condition']].head())


aphos sites: (34248, 11) (rows=sites, cols=samples+meta)
sn_classI:   (34636, 6)

aphos_wide: (6, 34249)   (rows=samples, cols=sites + condition)
sn_wide:    (6, 34637)

PTM_Collapse_key                                   condition
sample                                                      
20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dil...   withEGF
20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dil...   withEGF
20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dil...   withEGF
20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dil...     woEGF
20250729_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dil...     woEGF


In [23]:
# 1) Valid-values filter (≥70% present in at least one condition)
# 2) KNN imputation
# Dublin's filter_phosphosites identifies sites by the presence of '~' in the
# column name (alphaphos format). The SN-classI columns are like 'P12345|S|10|1'
# — we need to make them filterable by either tagging them or adapting the call.

# Dublin's filter_phosphosites looks at columns whose name contains '~'.
# For SN we replace '|' with '~' in the column names so the filter recognises
# them as site columns. We restore later if needed.

def normalize_site_col_names(df):
    rename = {c: c.replace('|', '~') for c in df.columns if '|' in str(c)}
    return df.rename(columns=rename)

aphos_wide_n = aphos_wide  # already has '~' in keys
sn_wide_n    = normalize_site_col_names(sn_wide)

print('aphos: sample of site-col names:', [c for c in aphos_wide_n.columns if '~' in str(c)][:2])
print('sn:    sample of site-col names:', [c for c in sn_wide_n.columns if '~' in str(c)][:2])

print('\nFiltering aphos ...')
aphos_filtered = filter_phosphosites(aphos_wide_n, how='condition', cutoff=0.7, condition_col='condition')
n_aphos_kept = sum('~' in str(c) for c in aphos_filtered.columns)
print(f'  aphos sites kept: {n_aphos_kept:,} (of {sum("~" in str(c) for c in aphos_wide_n.columns):,})')

print('\nFiltering sn ...')
sn_filtered = filter_phosphosites(sn_wide_n, how='condition', cutoff=0.7, condition_col='condition')
n_sn_kept = sum('~' in str(c) for c in sn_filtered.columns)
print(f'  sn sites kept:    {n_sn_kept:,} (of {sum("~" in str(c) for c in sn_wide_n.columns):,})')

print('\nImputing aphos ...')
aphos_imputed = impute_phosphosites(aphos_filtered)
print(f'  aphos_imputed: {aphos_imputed.shape}')

print('Imputing sn ...')
sn_imputed = impute_phosphosites(sn_filtered)
print(f'  sn_imputed:    {sn_imputed.shape}')


aphos: sample of site-col names: ['A0A087WUV0~ZNF892_S118_M1', 'A0A087WUV0~ZNF892_S122_M1']
sn:    sample of site-col names: ['A0A087WUV0~S~118~1', 'A0A087WUV0~S~122~1']

Filtering aphos ...
  aphos sites kept: 17,615 (of 34,248)

Filtering sn ...
  sn sites kept:    15,557 (of 34,636)

Imputing aphos ...
  aphos_imputed: (6, 17616)
Imputing sn ...
  sn_imputed:    (6, 15558)


In [24]:
import subprocess

LIMMA_WRAPPER = Path(r'D:/Projects/alphaPhos/docs/design/limma_wrapper.R')
RSCRIPT       = Path(r'C:/Program Files/R/R-4.5.2/bin/Rscript.exe')

LIMMA_OUT = OUT / 'limma'
LIMMA_OUT.mkdir(exist_ok=True)


def run_limma_via_rscript(imputed_wide, label):
    """
    imputed_wide: samples-as-rows DataFrame with a 'condition' column.
    Builds a features x samples expression matrix and a meta TSV,
    invokes the Rscript limma wrapper, returns the parsed topTable DataFrame.
    """
    site_cols = [c for c in imputed_wide.columns if '~' in str(c)]
    expr = imputed_wide[site_cols].T   # rows=features, cols=samples
    expr.index.name = 'feature'

    meta = pd.DataFrame({
        'sample_id': imputed_wide.index,
        'treatment': imputed_wide['condition'].values,
    })

    expr_path = LIMMA_OUT / f'{label}_expr.tsv'
    meta_path = LIMMA_OUT / f'{label}_meta.tsv'
    out_path  = LIMMA_OUT / f'{label}_topTable.tsv'

    expr.to_csv(expr_path, sep=chr(9), na_rep='NA')
    meta.to_csv(meta_path, sep=chr(9), index=False)

    cmd = [str(RSCRIPT), str(LIMMA_WRAPPER), str(expr_path), str(meta_path),
           'treatment', str(out_path)]
    print(f'  [{label}] running limma ...')
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print('  STDERR:', result.stderr[-500:])
        raise RuntimeError(f'limma failed for {label}')

    res = pd.read_csv(out_path, sep=chr(9))
    P_CUT, LFC_CUT = 0.05, 0.585
    res['significant'] = (res['adj.P.Val'] < P_CUT) & (res['logFC'].abs() > LFC_CUT)
    return res


print('==== limma: aphos_sum_classI ====')
limma_aphos = run_limma_via_rscript(aphos_imputed, 'aphos_sum_classI')
print(f'  features: {len(limma_aphos):,}')
print(f'  significant (adj.P<0.05 & |logFC|>0.585): {int(limma_aphos["significant"].sum())}')
print(f'  raw P<0.05: {int((limma_aphos["P.Value"] < 0.05).sum())}')

print('\n==== limma: sn_classI ====')
limma_sn = run_limma_via_rscript(sn_imputed, 'sn_classI')
print(f'  features: {len(limma_sn):,}')
print(f'  significant (adj.P<0.05 & |logFC|>0.585): {int(limma_sn["significant"].sum())}')
print(f'  raw P<0.05: {int((limma_sn["P.Value"] < 0.05).sum())}')


==== limma: aphos_sum_classI ====
  [aphos_sum_classI] running limma ...
  features: 17,615
  significant (adj.P<0.05 & |logFC|>0.585): 2097
  raw P<0.05: 5380

==== limma: sn_classI ====
  [sn_classI] running limma ...
  features: 15,557
  significant (adj.P<0.05 & |logFC|>0.585): 1921
  raw P<0.05: 4768


In [25]:
# Compare the two limma results: significant-hit overlap + logFC concordance

# To match cross-version, normalize feature IDs to a common scheme. alphaphos
# features are 'P12345;Q67890~GENE_S15_M1', SN features are 'P12345~S~15~1'.
# We collapse both to (first_protein, AA, position, mult) tuples.

def alpha_feat_to_matchkey(f):
    s = str(f)
    if '~' not in s:
        return None
    # alphaphos style: 'P12345;Q67890~GENE_S15_M1'
    head, tail = s.split('~', 1)
    first_prot = head.split(';')[0]
    # GENE_S15_M1
    m = re.match(r'^[^_]*_([A-Z])(\d+)_M(\d+)$', tail)
    if not m:
        return None
    return f'{first_prot}|{m.group(1)}|{int(m.group(2))}|{int(m.group(3))}'


def sn_feat_to_matchkey(f):
    s = str(f)
    if s.count('~') == 3:                      # SN style after our '|' -> '~' rename
        prot, aa, pos, mult = s.split('~')
        return f'{prot}|{aa}|{int(pos)}|{int(mult)}'
    if s.count('|') == 3:
        return s
    return None


limma_aphos['match_key'] = limma_aphos['feature'].map(alpha_feat_to_matchkey)
limma_sn['match_key']    = limma_sn['feature'].map(sn_feat_to_matchkey)

n_a_unmapped = int(limma_aphos['match_key'].isna().sum())
n_s_unmapped = int(limma_sn['match_key'].isna().sum())
print(f'unmapped feature ids: aphos={n_a_unmapped}, sn={n_s_unmapped}')

# Significant-hit overlap
sig_a = set(limma_aphos.loc[limma_aphos['significant'], 'match_key'].dropna())
sig_s = set(limma_sn.loc[limma_sn['significant'], 'match_key'].dropna())
shared_sig  = sig_a & sig_s
aphos_only  = sig_a - sig_s
sn_only     = sig_s - sig_a
jaccard_sig = len(shared_sig) / max(1, len(sig_a | sig_s))

print()
print(f'Significant hits (adj.P<0.05 & |logFC|>0.585):')
print(f'  aphos sig:      {len(sig_a):>5,}')
print(f'  sn sig:         {len(sig_s):>5,}')
print(f'  shared:         {len(shared_sig):>5,}')
print(f'  aphos-only:     {len(aphos_only):>5,}')
print(f'  sn-only:        {len(sn_only):>5,}')
print(f'  Jaccard:        {jaccard_sig:.3f}')

# logFC concordance on all features in BOTH limma tables
lfc = (limma_aphos[['match_key', 'logFC', 'P.Value', 'adj.P.Val']]
       .merge(limma_sn[['match_key', 'logFC', 'P.Value', 'adj.P.Val']],
              on='match_key', suffixes=('_aphos', '_sn'))
       .dropna(subset=['logFC_aphos', 'logFC_sn']))

pear = float(lfc['logFC_aphos'].corr(lfc['logFC_sn']))
spear = float(lfc['logFC_aphos'].corr(lfc['logFC_sn'], method='spearman'))
print()
print(f'logFC concordance on {len(lfc):,} shared features:')
print(f'  Pearson r:  {pear:.4f}')
print(f'  Spearman r: {spear:.4f}')
print(f'  median |diff|: {(lfc["logFC_aphos"] - lfc["logFC_sn"]).abs().median():.4f}')

# Top-15 hits per version, side by side
top_a = (limma_aphos[limma_aphos['significant']]
         .nsmallest(15, 'adj.P.Val')[['match_key', 'logFC', 'adj.P.Val']]
         .reset_index(drop=True))
top_a.columns = ['match_key', 'aphos_logFC', 'aphos_adjP']

top_s = (limma_sn[limma_sn['significant']]
         .nsmallest(15, 'adj.P.Val')[['match_key', 'logFC', 'adj.P.Val']]
         .reset_index(drop=True))
top_s.columns = ['match_key', 'sn_logFC', 'sn_adjP']

print()
print('Top 15 hits per version:')
print(pd.concat({'aphos': top_a, 'sn': top_s}, axis=1).to_string())


unmapped feature ids: aphos=0, sn=0

Significant hits (adj.P<0.05 & |logFC|>0.585):
  aphos sig:      2,097
  sn sig:         1,921
  shared:         1,548
  aphos-only:       549
  sn-only:          373
  Jaccard:        0.627

logFC concordance on 15,281 shared features:
  Pearson r:  0.8780
  Spearman r: 0.8726
  median |diff|: 0.0363

Top 15 hits per version:
              aphos                                      sn                    
          match_key aphos_logFC aphos_adjP        match_key  sn_logFC   sn_adjP
0    O15027|S|589|3    3.350182   0.000121  P00533|Y|1172|1 -6.638459  0.000059
1   P00533|Y|1172|1   -6.586553   0.000121    P46108|S|74|1 -5.377655  0.000059
2   P12270|S|2155|1   -1.630194   0.000121   Q68EM7|S|702|1 -1.868355  0.000059
3    P49757|T|363|1   -5.487088   0.000121    P04792|S|82|2 -2.170602  0.000100
4    P51858|S|107|1   -2.472222   0.000121    P04792|S|78|2 -2.170602  0.000100
5    Q15418|S|363|1    3.506383   0.000121  P12270|S|2155|1 -1.642378  0.0

In [26]:
import plotly.express as px
import plotly.graph_objects as go

# Mark each shared feature by joint significance class for color
def _class(row):
    is_a = row['adj.P.Val_aphos'] < 0.05 and abs(row['logFC_aphos']) > 0.585
    is_s = row['adj.P.Val_sn']    < 0.05 and abs(row['logFC_sn'])    > 0.585
    if is_a and is_s: return 'both_sig'
    if is_a:           return 'aphos_only'
    if is_s:           return 'sn_only'
    return 'not_sig'

lfc['class'] = lfc.apply(_class, axis=1)
print(lfc['class'].value_counts())

# 1) logFC vs logFC scatter
fig1 = px.scatter(
    lfc, x='logFC_sn', y='logFC_aphos', color='class',
    color_discrete_map={'both_sig': '#e60000', 'aphos_only': '#1f77b4',
                        'sn_only': '#ff7f0e',  'not_sig':   '#cccccc'},
    opacity=0.5,
    title=f'logFC concordance — Pearson r = {pear:.4f}',
    labels={'logFC_sn': 'sn_classI logFC (withEGF - woEGF)',
            'logFC_aphos': 'alphaphos+sum logFC (withEGF - woEGF)'},
    hover_data={'match_key': True, 'adj.P.Val_aphos': ':.3g', 'adj.P.Val_sn': ':.3g'},
)
mn, mx = float(lfc[['logFC_aphos','logFC_sn']].min().min()), float(lfc[['logFC_aphos','logFC_sn']].max().max())
fig1.add_shape(type='line', x0=mn, x1=mx, y0=mn, y1=mx, line=dict(dash='dash', color='black', width=1))
fig1.update_layout(height=550, width=750, template='plotly_white')
fig1.show()

# 2) Volcano for each version side-by-side
fig2 = go.Figure()
for label, df, color in [('aphos+sum', limma_aphos, '#e60000'), ('sn_classI', limma_sn, '#1f77b4')]:
    d = df.dropna(subset=['logFC', 'adj.P.Val']).copy()
    d['nlog10p'] = -np.log10(d['adj.P.Val'].clip(lower=1e-300))
    d['is_sig'] = d['significant']
    fig2.add_trace(go.Scatter(
        x=d['logFC'], y=d['nlog10p'],
        mode='markers', name=label,
        marker=dict(size=5, color=color,
                    opacity=np.where(d['is_sig'], 0.85, 0.18)),
        hovertext=d['feature'], hoverinfo='text+x+y',
    ))
fig2.add_hline(y=-np.log10(0.05), line_dash='dash', line_color='black')
fig2.add_vline(x=0.585,  line_dash='dot',  line_color='black')
fig2.add_vline(x=-0.585, line_dash='dot',  line_color='black')
fig2.update_layout(
    title='Volcano — withEGF vs woEGF', xaxis_title='logFC',
    yaxis_title='-log10(adj.P.Val)',
    height=500, width=900, template='plotly_white',
)
fig2.show()


class
not_sig       13056
both_sig       1548
sn_only         339
aphos_only      338
Name: count, dtype: int64


## §8 · Same filtering policy on both sides

In §7 we compared `aphos_only_MS2_sum_classI` against `sn_classI`. But those
two have different filtering policies:

- alphaphos: per-condition majority rule (`apply_condition_aware_classI_mask`,
  default threshold 0.50)
- SN classI: per-cell binary threshold (Spectronaut's published 0.75 filter)

To isolate the *algorithm* (sum vs SN consolidation) from the *filtering policy*,
we re-do the comparison after applying **the same condition-aware mask** to
`sn_all` (Spectronaut's unfiltered PTM site report). This puts both versions
on identical filtering ground; remaining differences come from the collapse
itself.


In [27]:
# Convert sn_all to alphaphos-compatible site matrix and apply the mask.
sn_all_for_mask = sn_all_sites.copy()
sn_all_for_mask['PTM_Collapse_key'] = sn_all_for_mask.index
sn_all_for_mask = sn_all_for_mask.reset_index(drop=True)

print(f'sn_all (pre-mask): {len(sn_all_for_mask):,} sites')

sn_all_classI_aware, sn_decision = apply_condition_aware_classI_mask(
    df_sites=sn_all_for_mask,
    loc_per_run=sn_all_loc,
    sample_to_condition=condition_df,
    classI_cutoff=0.75,
    condition_threshold=0.50,
    drop_all_nan=True,
    return_decision_table=True,
)
print(f'sn_all + condition_aware: {len(sn_all_classI_aware):,} sites')
print(f'sn_classI (binary):       {len(sn_classI_sites):,} sites for reference')

# Build the samples-as-rows form for the downstream pipeline (matching §7)
sn_aware_wide = (sn_all_classI_aware.set_index('PTM_Collapse_key')
                                    [condition_df['sample'].tolist()].T)
sn_aware_wide.index.name = 'sample'
sn_aware_wide['condition'] = sn_aware_wide.index.map(s2c_map)

# Match the renaming we did in §7 (replace '|' with '~' so filter_phosphosites recognises site cols)
sn_aware_wide_n = normalize_site_col_names(sn_aware_wide)
print(f'\nsn_aware_wide_n: {sn_aware_wide_n.shape}  (rows=samples, cols=sites+condition)')


sn_all (pre-mask): 66,410 sites
sn_all + condition_aware: 35,695 sites
sn_classI (binary):       34,636 sites for reference

sn_aware_wide_n: (6, 35696)  (rows=samples, cols=sites+condition)


In [28]:
# Filter + impute + limma — identical settings to §7
print('Filtering ...')
sn_aware_filtered = filter_phosphosites(sn_aware_wide_n, how='condition',
                                         cutoff=0.7, condition_col='condition')
n_kept = sum('~' in str(c) for c in sn_aware_filtered.columns)
print(f'  sites kept: {n_kept:,}')

print('Imputing ...')
sn_aware_imputed = impute_phosphosites(sn_aware_filtered)
print(f'  shape: {sn_aware_imputed.shape}')

print('Running limma ...')
limma_sn_aware = run_limma_via_rscript(sn_aware_imputed, 'sn_all_condition_aware')
print(f'  features: {len(limma_sn_aware):,}')
print(f'  significant (adj.P<0.05 & |logFC|>0.585): {int(limma_sn_aware["significant"].sum())}')
print(f'  raw P<0.05: {int((limma_sn_aware["P.Value"] < 0.05).sum())}')


Filtering ...
  sites kept: 23,144
Imputing ...
  shape: (6, 23145)
Running limma ...
  [sn_all_condition_aware] running limma ...
  features: 23,144
  significant (adj.P<0.05 & |logFC|>0.585): 4077
  raw P<0.05: 9241


In [29]:
# Three-way compare: aphos+sum+classI vs sn_classI (§7) vs sn_all+condition_aware (§8)

limma_sn_aware['match_key'] = limma_sn_aware['feature'].map(sn_feat_to_matchkey)
print(f'unmapped sn_aware feature ids: {int(limma_sn_aware["match_key"].isna().sum())}')

sig_aware = set(limma_sn_aware.loc[limma_sn_aware['significant'], 'match_key'].dropna())
print()
print('Significant-hit overlap:')
print(f'  aphos sig:                  {len(sig_a):>5,}')
print(f'  sn_classI sig (§7):         {len(sig_s):>5,}')
print(f'  sn_all+cond_aware sig (§8): {len(sig_aware):>5,}')
print()
print(f'  aphos ∩ sn_classI       : {len(sig_a & sig_s):>5,}  J = {len(sig_a & sig_s)/max(1,len(sig_a | sig_s)):.3f}')
print(f'  aphos ∩ sn_all+aware    : {len(sig_a & sig_aware):>5,}  J = {len(sig_a & sig_aware)/max(1,len(sig_a | sig_aware)):.3f}')
print(f'  sn_classI ∩ sn_all+aware: {len(sig_s & sig_aware):>5,}  J = {len(sig_s & sig_aware)/max(1,len(sig_s | sig_aware)):.3f}')

# logFC concordance: aphos vs sn_all+aware
lfc_aware = (limma_aphos[['match_key', 'logFC', 'P.Value', 'adj.P.Val']]
             .merge(limma_sn_aware[['match_key', 'logFC', 'P.Value', 'adj.P.Val']],
                    on='match_key', suffixes=('_aphos', '_snaware'))
             .dropna(subset=['logFC_aphos', 'logFC_snaware']))

pear_aware  = float(lfc_aware['logFC_aphos'].corr(lfc_aware['logFC_snaware']))
spear_aware = float(lfc_aware['logFC_aphos'].corr(lfc_aware['logFC_snaware'], method='spearman'))
med_diff_aware = float((lfc_aware['logFC_aphos'] - lfc_aware['logFC_snaware']).abs().median())

print()
print(f'logFC concordance (aphos+sum vs sn_all+aware) on {len(lfc_aware):,} shared features:')
print(f'  Pearson r:    {pear_aware:.4f}   (was {pear:.4f} vs sn_classI)')
print(f'  Spearman r:   {spear_aware:.4f}   (was {spear:.4f} vs sn_classI)')
print(f'  median |diff|: {med_diff_aware:.4f}  (was {(lfc["logFC_aphos"] - lfc["logFC_sn"]).abs().median():.4f})')

print()
print('Summary:')
print(f'  Filtering policy:   §7 used SN binary, §8 uses condition_aware on both sides')
print(f'  Significant overlap:  Jaccard {len(sig_a & sig_s)/max(1,len(sig_a | sig_s)):.3f} -> {len(sig_a & sig_aware)/max(1,len(sig_a | sig_aware)):.3f}')
print(f'  logFC Pearson r:      {pear:.4f}            -> {pear_aware:.4f}')


unmapped sn_aware feature ids: 0

Significant-hit overlap:
  aphos sig:                  2,097
  sn_classI sig (§7):         1,921
  sn_all+cond_aware sig (§8): 4,077

  aphos ∩ sn_classI       : 1,548  J = 0.627
  aphos ∩ sn_all+aware    : 1,809  J = 0.414
  sn_classI ∩ sn_all+aware: 1,628  J = 0.373

logFC concordance (aphos+sum vs sn_all+aware) on 17,614 shared features:
  Pearson r:    0.7488   (was 0.8780 vs sn_classI)
  Spearman r:   0.7716   (was 0.8726 vs sn_classI)
  median |diff|: 0.0350  (was 0.0363)

Summary:
  Filtering policy:   §7 used SN binary, §8 uses condition_aware on both sides
  Significant overlap:  Jaccard 0.627 -> 0.414
  logFC Pearson r:      0.8780            -> 0.7488


In [30]:
# Updated logFC scatter: aphos+sum vs sn_all+condition_aware
def _class2(row):
    is_a = row['adj.P.Val_aphos']  < 0.05 and abs(row['logFC_aphos'])  > 0.585
    is_s = row['adj.P.Val_snaware'] < 0.05 and abs(row['logFC_snaware']) > 0.585
    if is_a and is_s: return 'both_sig'
    if is_a:           return 'aphos_only'
    if is_s:           return 'sn_aware_only'
    return 'not_sig'

lfc_aware['class'] = lfc_aware.apply(_class2, axis=1)
print(lfc_aware['class'].value_counts())

import plotly.express as px
fig = px.scatter(
    lfc_aware, x='logFC_snaware', y='logFC_aphos', color='class',
    color_discrete_map={'both_sig': '#e60000', 'aphos_only': '#1f77b4',
                        'sn_aware_only': '#ff7f0e', 'not_sig': '#cccccc'},
    opacity=0.5,
    title=f'logFC concordance (same filtering policy) — Pearson r = {pear_aware:.4f}',
    labels={'logFC_snaware': 'sn_all+condition_aware logFC',
            'logFC_aphos':   'alphaphos+sum logFC'},
    hover_data={'match_key': True, 'adj.P.Val_aphos': ':.3g', 'adj.P.Val_snaware': ':.3g'},
)
mn = float(lfc_aware[['logFC_aphos','logFC_snaware']].min().min())
mx = float(lfc_aware[['logFC_aphos','logFC_snaware']].max().max())
fig.add_shape(type='line', x0=mn, x1=mx, y0=mn, y1=mx, line=dict(dash='dash', color='black', width=1))
fig.update_layout(height=550, width=750, template='plotly_white')
fig.show()


class
not_sig          14272
both_sig          1809
sn_aware_only     1245
aphos_only         288
Name: count, dtype: int64


## §9 · Sample QC — PCA + per-sample CV

Two technical checks comparing `aphos_imputed` and `sn_imputed` (the post-filter,
post-imputation matrices that fed into limma in §7):

1. **PCA** — Z-score each site across samples, PCA on samples × sites.
   Do withEGF / woEGF replicates separate the same way in both versions?
2. **Per-site CV within condition** — distribution of linear-space CV across
   replicates. Tighter CV = better technical reproducibility.

If both versions show clean condition separation in PC1 and comparable CV
distributions, technical reproducibility is equivalent.


In [31]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def run_pca(imputed_wide, label):
    site_cols = [c for c in imputed_wide.columns if '~' in str(c)]
    X = imputed_wide[site_cols].values
    X_z = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2)
    scores = pca.fit_transform(X_z)
    var = pca.explained_variance_ratio_ * 100
    return scores, var


aphos_pca, aphos_var = run_pca(aphos_imputed, 'aphos')
sn_pca,    sn_var    = run_pca(sn_imputed, 'sn')

aphos_meta = aphos_imputed[['condition']].copy()
aphos_meta[['PC1', 'PC2']] = aphos_pca
sn_meta = sn_imputed[['condition']].copy()
sn_meta[['PC1', 'PC2']]    = sn_pca

print(f'aphos PC1 / PC2 explained variance: {aphos_var[0]:.1f}% / {aphos_var[1]:.1f}%')
print(f'sn    PC1 / PC2 explained variance: {sn_var[0]:.1f}% / {sn_var[1]:.1f}%')

# Plot side-by-side
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f'aphos+sum+classI ({aphos_var[0]:.1f}% / {aphos_var[1]:.1f}%)',
    f'sn_classI ({sn_var[0]:.1f}% / {sn_var[1]:.1f}%)',
])

palette = {'withEGF': '#e60000', 'woEGF': '#1f77b4'}
for col, (df, n_sites) in enumerate([(aphos_meta, len([c for c in aphos_imputed.columns if '~' in str(c)])),
                                      (sn_meta,    len([c for c in sn_imputed.columns    if '~' in str(c)]))], 1):
    for cond, sub in df.groupby('condition'):
        fig.add_trace(go.Scatter(
            x=sub['PC1'], y=sub['PC2'],
            mode='markers',
            marker=dict(size=14, color=palette.get(cond, '#888'), line=dict(width=1, color='black')),
            name=f'{cond} (n={len(sub)})',
            text=sub.index, hoverinfo='text+x+y',
            showlegend=(col == 1),
        ), row=1, col=col)
    fig.update_xaxes(title_text='PC1', row=1, col=col)
    fig.update_yaxes(title_text='PC2', row=1, col=col)

fig.update_layout(height=450, width=1000, template='plotly_white',
                  title='PCA — same samples, two pipeline versions')
fig.show()


aphos PC1 / PC2 explained variance: 42.6% / 20.7%
sn    PC1 / PC2 explained variance: 42.6% / 21.0%


In [32]:
# Per-site CV within condition (linear space) — distribution comparison

def per_site_cv_per_condition(imputed_wide, label):
    site_cols = [c for c in imputed_wide.columns if '~' in str(c)]
    rows = []
    for cond, sub in imputed_wide.groupby('condition'):
        if cond not in ('withEGF', 'woEGF'):
            continue
        block = sub[site_cols]
        lin = np.power(2.0, block)
        mean = lin.mean(axis=0, skipna=True)
        sd = lin.std(axis=0, skipna=True, ddof=1)
        cv = (sd / mean * 100).dropna()
        for v in cv.values:
            rows.append({'version': label, 'condition': cond, 'cv_pct': v})
    return pd.DataFrame(rows)


cv_aphos = per_site_cv_per_condition(aphos_imputed, 'aphos+sum+classI')
cv_sn    = per_site_cv_per_condition(sn_imputed, 'sn_classI')
cv_all = pd.concat([cv_aphos, cv_sn], ignore_index=True)

print('Per-site CV summary (%):')
print(cv_all.groupby(['version', 'condition'])['cv_pct'].describe(percentiles=[0.25, 0.5, 0.75]).round(1))

import plotly.express as px
y_top = float(np.nanpercentile(cv_all['cv_pct'], 99))
fig = px.box(
    cv_all, x='condition', y='cv_pct', color='version',
    points=False,
    category_orders={'condition': ['woEGF', 'withEGF'], 'version': ['aphos+sum+classI', 'sn_classI']},
    title='Per-site CV within condition (linear space)',
    color_discrete_map={'aphos+sum+classI': '#e60000', 'sn_classI': '#1f77b4'},
)
fig.update_traces(boxmean=True)
fig.update_yaxes(title='CV %', range=[0, y_top])
fig.update_layout(height=480, width=800, template='plotly_white')
fig.show()


Per-site CV summary (%):
                              count  mean   std  min  25%   50%   75%    max
version          condition                                                  
aphos+sum+classI withEGF    17615.0  20.8  23.0  0.0  5.5  11.7  27.4  172.0
                 woEGF      17615.0  24.8  24.9  0.1  7.9  15.6  32.7  167.2
sn_classI        withEGF    15557.0  19.7  22.5  0.1  5.3  10.9  24.8  170.5
                 woEGF      15557.0  24.0  24.4  0.0  7.7  15.0  31.2  167.2


In [33]:
# One-glance summary
print('=== Sample QC summary ===')
print()
print(f'aphos+sum+classI:  {sum("~" in str(c) for c in aphos_imputed.columns):,} sites, '
      f'PC1 {aphos_var[0]:.1f}% / PC2 {aphos_var[1]:.1f}%, '
      f'median within-condition CV: {cv_aphos["cv_pct"].median():.1f}%')
print(f'sn_classI:         {sum("~" in str(c) for c in sn_imputed.columns):,} sites, '
      f'PC1 {sn_var[0]:.1f}% / PC2 {sn_var[1]:.1f}%, '
      f'median within-condition CV: {cv_sn["cv_pct"].median():.1f}%')
print()
print('If PC1 explains a similar % of variance and median CV is similar,')
print('technical reproducibility is equivalent across both versions.')


=== Sample QC summary ===

aphos+sum+classI:  17,615 sites, PC1 42.6% / PC2 20.7%, median within-condition CV: 13.7%
sn_classI:         15,557 sites, PC1 42.6% / PC2 21.0%, median within-condition CV: 12.8%

If PC1 explains a similar % of variance and median CV is similar,
technical reproducibility is equivalent across both versions.
